# Perception reconstructions

Load a trained `PerceptionAutoencoder`, reconstruct real Crafter frames, and
plot fidelity metrics + per-pixel error heatmaps inline.

Defaults to `checkpoints/m1_autoencoder_finetune/ckpt_best.pt` and
`data/m1_random_frames.pt` if present.


In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import yaml
from torch import Tensor

from models.autoencoder import PerceptionAutoencoder
from models.preprocess import nchw_float_to_nhwc_uint8, nhwc_uint8_to_nchw_float
from training.device import get_device

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

%matplotlib inline

CONFIG = Path("configs/m1_autoencoder_finetune.yaml")
CKPT = Path("checkpoints/m1_autoencoder_finetune/ckpt_best.pt")
N_SHOW = 6

device = get_device()
print(f"device: {device}")
print(f"ckpt exists: {CKPT.exists()}  frames will load from config collect.out_path")


## Load model + frames


In [ ]:
with CONFIG.open() as f:
    cfg = yaml.safe_load(f)

frames_path = Path(cfg["collect"]["out_path"])
if not frames_path.exists():
    raise FileNotFoundError(
        f"Missing {frames_path}. Run: python scripts/collect_random_frames.py"
    )
if not CKPT.exists():
    raise FileNotFoundError(f"Missing {CKPT}. Train or point CKPT at another file.")

frames: Tensor = torch.load(frames_path, weights_only=False)["frames"]
print(f"frames: {tuple(frames.shape)} from {frames_path}")

model = PerceptionAutoencoder(
    embed_dim=int(cfg["embed_dim"]),
    channels=tuple(int(c) for c in cfg.get("encoder_channels", [64, 128, 256, 512])),
    stem_channels=int(cfg.get("stem_channels", 64)),
).to(device)
ckpt = torch.load(CKPT, weights_only=False, map_location=device)
model.load_state_dict(ckpt["model"], strict=True)
model.eval()
print(f"loaded {CKPT} (ckpt step={int(ckpt.get('step', -1))})")


## Reconstruct a diverse batch

Pick frames that are spread out in mean-RGB space so the grid isn't all the same biome.


In [ ]:
def select_diverse_frames(frames: Tensor, n: int) -> Tensor:
    if n >= frames.shape[0]:
        return frames.clone()
    flat = frames.float().mean(dim=(1, 2))
    chosen = [int(frames.shape[0] // 5)]
    for _ in range(n - 1):
        refs = flat[chosen]
        d = torch.cdist(flat, refs).min(dim=1).values
        d[chosen] = -1.0
        chosen.append(int(torch.argmax(d).item()))
    return frames[chosen].clone()


def uint8_fidelity(real_u8: Tensor, recon_u8: Tensor) -> dict[str, float]:
    diff = (real_u8.float() - recon_u8.float()).abs()
    channel_max = diff.amax(dim=-1)
    return {
        "uint8_mad": float(diff.mean().item()),
        "within1": float(channel_max.le(1).float().mean().item()),
        "within2": float(channel_max.le(2).float().mean().item()),
        "exact": float(channel_max.eq(0).float().mean().item()),
    }


batch_u8 = select_diverse_frames(frames, N_SHOW)
with torch.no_grad():
    obs = nhwc_uint8_to_nchw_float(batch_u8.to(device))
    recon, embed = model(obs)
    real_u8 = nchw_float_to_nhwc_uint8(obs.cpu())
    recon_u8 = nchw_float_to_nhwc_uint8(recon.cpu())

metrics = {
    "recon_mse": float(F.mse_loss(recon, obs).item()),
    "recon_l1": float(F.l1_loss(recon, obs).item()),
    **uint8_fidelity(real_u8, recon_u8),
}
print("batch metrics:")
for k, v in metrics.items():
    print(f"  {k:12s} {v:.6f}")
print(f"embed shape: {tuple(embed.shape)}")


## Side-by-side: real | recon | |error|


In [ ]:
err = (real_u8.float() - recon_u8.float()).abs().mean(dim=-1).numpy()

fig, axes = plt.subplots(N_SHOW, 3, figsize=(9, 2.4 * N_SHOW))
if N_SHOW == 1:
    axes = np.expand_dims(axes, 0)

for i in range(N_SHOW):
    axes[i, 0].imshow(real_u8[i].numpy())
    axes[i, 0].set_ylabel(f"#{i}", rotation=0, labelpad=18, va="center")
    axes[i, 1].imshow(recon_u8[i].numpy())
    im = axes[i, 2].imshow(err[i], cmap="magma", vmin=0, vmax=max(8.0, float(err.max())))
    for ax, title in zip(axes[i], ("real", "recon", "|err| mean-RGB")):
        ax.set_xticks([])
        ax.set_yticks([])
        if i == 0:
            ax.set_title(title)

fig.colorbar(im, ax=axes[:, 2].ravel().tolist(), fraction=0.03, pad=0.02, label="mean |Δ| (0–255)")
fig.suptitle("Perception AE reconstructions", y=1.01)
fig.tight_layout()
plt.show()


## Metric bars + error histogram


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

names = ["within1", "within2", "exact"]
vals = [metrics[n] for n in names]
axes[0].bar(names, vals, color=["#2a9d8f", "#457b9d", "#1d3557"])
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel("fraction of pixels")
axes[0].set_title("uint8 fidelity (this batch)")
for x, v in zip(names, vals):
    axes[0].text(x, v + 0.02, f"{v:.3f}", ha="center", fontsize=9)

flat_err = (real_u8.float() - recon_u8.float()).abs().numpy().ravel()
axes[1].hist(flat_err, bins=40, color="#e76f51", alpha=0.9)
axes[1].axvline(metrics["uint8_mad"], color="black", linestyle="--", label=f"MAD={metrics['uint8_mad']:.3f}")
axes[1].set_xlabel("|Δ| gray levels")
axes[1].set_ylabel("count")
axes[1].set_title("per-channel absolute error")
axes[1].legend()

fig.tight_layout()
plt.show()


## Optional: larger eval sample

Bump `N_EVAL` if you want metrics closer to the full-dataset numbers from training.


In [ ]:
N_EVAL = 256
idx = torch.randperm(frames.shape[0])[:N_EVAL]
eval_u8 = frames[idx]

with torch.no_grad():
    obs = nhwc_uint8_to_nchw_float(eval_u8.to(device))
    recon, _ = model(obs)
    real_u8 = nchw_float_to_nhwc_uint8(obs.cpu())
    recon_u8 = nchw_float_to_nhwc_uint8(recon.cpu())

eval_metrics = uint8_fidelity(real_u8, recon_u8)
print(f"eval on {N_EVAL} random frames:")
for k, v in eval_metrics.items():
    print(f"  {k:12s} {v:.6f}")

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(list(eval_metrics.keys()), list(eval_metrics.values()), color="#264653")
ax.set_title(f"uint8 metrics on {N_EVAL} frames")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()
